In [1]:
import os
import json
import time
from dotenv import load_dotenv
from huggingface_hub import login
from price_agent.data.items import Item
from price_agent.data.evaluator import evaluate
from google.cloud import storage
import vertexai
from vertexai.tuning import sft

d:\ujjwal\the_capstone_project\.venv\Lib\site-packages\huggingface_hub\constants.py:310: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
d:\ujjwal\the_capstone_project\.venv\Lib\site-packages\google\cloud\aiplatform\models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


In [2]:
# ----------------------------------------------------
# 1. Environment & Hugging Face Setup
# ----------------------------------------------------
LITE_MODE = True

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

username = "ujjwalsingh108"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

fine_tune_train = train[:100]
fine_tune_validation = val[:50]

2026-08-22 16:22:00,303 INFO httpx HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-08-22 16:22:00,709 INFO httpx HTTP Request: GET https://huggingface.co/api/whoami-v2 "HTTP/1.1 200 OK"
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
2026-08-22 16:22:01,060 WARNING huggingface_hub._login Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
2026-08-22 16:22:01,426 INFO httpx HTTP Request: HEAD https://huggingface.co/datasets/ujjwalsingh108/items_lite/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-08-22 16:22:01,467 INFO httpx HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/ujjwalsingh108/items_lite/457bd961c240529a681fe965e44297f6d98a19a8/README.md "HTTP/1.1 200 OK"
2026-08-22 16:22:01,737 INFO httpx HTTP Request: HEAD https://huggingface.co/datasets/ujjwalsingh

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


In [3]:
# ----------------------------------------------------
# 2. Format Data for Gemini (Contents & Parts structure)
# ----------------------------------------------------
def messages_for_gemini(item: Item):
    prompt = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    target = f"${item.price:.3f}"
    return {
        "contents": [
            {"role": "user", "parts": [{"text": prompt}]},
            {"role": "model", "parts": [{"text": target}]}
        ]
    }

def write_gemini_jsonl(items: list[Item], filename: str):
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    with open(filename, "w") as f:
        for item in items:
            f.write(json.dumps(messages_for_gemini(item)) + "\n")

train_file_path = "../../data/04-predictions/fine_tune_train.jsonl"
val_file_path = "../../data/04-predictions/fine_tune_validation.jsonl"

write_gemini_jsonl(fine_tune_train, train_file_path)
write_gemini_jsonl(fine_tune_validation, val_file_path)
print("Saved local Gemini-formatted JSONL files.")

Saved local Gemini-formatted JSONL files.


## Dataset Size & Cost Considerations

OpenAI typically recommends starting fine-tuning with a small, high-quality dataset of **50–100 examples**.

For this project, I chose to scale up to **20,000 data points**.

### Key Takeaways

* **Total Cost:** Scaling up to 20,000 examples cost approximately **$3.42**.
* **Recommendation:** If you want to keep costs minimal while testing initial performance, stick to the recommended baseline of **100 examples**.

In [4]:
# ----------------------------------------------------
# 3. Upload to Google Cloud Storage (GCS)
# ----------------------------------------------------
PROJECT_ID = "starlit-primacy-480312-u9"  # Your GCP Project ID from screenshots
REGION = "us-central1"
BUCKET_NAME = f"{PROJECT_ID}-tuning-bucket"

# Initialize GCS client
storage_client = storage.Client(project=PROJECT_ID)

# Create bucket if it doesn't exist
try:
    bucket = storage_client.get_bucket(BUCKET_NAME)
except Exception:
    bucket = storage_client.create_bucket(BUCKET_NAME, location=REGION)
    print(f"Created bucket: {BUCKET_NAME}")

def upload_to_gcs(local_file, gcs_blob_name):
    blob = bucket.blob(gcs_blob_name)
    blob.upload_from_filename(local_file)
    return f"gs://{BUCKET_NAME}/{gcs_blob_name}"

train_gcs_uri = upload_to_gcs(train_file_path, "datasets/train.jsonl")
val_gcs_uri = upload_to_gcs(val_file_path, "datasets/validation.jsonl")

print(f"Train GCS URI: {train_gcs_uri}")
print(f"Val GCS URI: {val_gcs_uri}")

2026-08-22 16:24:42,959 WARNING google.auth._default No project ID could be determined. Consider running `gcloud config set project` or setting the GOOGLE_CLOUD_PROJECT environment variable


Created bucket: starlit-primacy-480312-u9-tuning-bucket
Train GCS URI: gs://starlit-primacy-480312-u9-tuning-bucket/datasets/train.jsonl
Val GCS URI: gs://starlit-primacy-480312-u9-tuning-bucket/datasets/validation.jsonl


In [5]:
# ----------------------------------------------------
# 4. Launch Fine-Tuning Job on Vertex AI
# ----------------------------------------------------
vertexai.init(project=PROJECT_ID, location=REGION)

print("Starting supervised fine-tuning for gemini-2.5-flash-lite...")
tuning_job = sft.train(
    source_model="gemini-2.5-flash-lite",
    train_dataset=train_gcs_uri,
    validation_dataset=val_gcs_uri,
    epochs=3,
    adapter_size=8,
    tuned_model_display_name="gemini-price-estimator-v1"
)

print(f"Tuning job started: {tuning_job.resource_name}")
print(f"Job state: {tuning_job.state}")

2026-08-22 16:25:08,523 INFO vertexai.tuning._tuning Creating SupervisedTuningJob


Starting supervised fine-tuning for gemini-2.5-flash-lite...


2026-08-22 16:25:12,135 INFO vertexai.tuning._tuning SupervisedTuningJob created. Resource name: projects/966750849516/locations/us-central1/tuningJobs/1206289746024726528
2026-08-22 16:25:12,139 INFO vertexai.tuning._tuning To use this SupervisedTuningJob in another session:
2026-08-22 16:25:12,139 INFO vertexai.tuning._tuning tuning_job = sft.SupervisedTuningJob('projects/966750849516/locations/us-central1/tuningJobs/1206289746024726528')
2026-08-22 16:25:12,140 INFO vertexai.tuning._tuning View Tuning Job:
https://console.cloud.google.com/vertex-ai/generative/language/locations/us-central1/tuning/tuningJob/1206289746024726528?project=966750849516


Tuning job started: projects/966750849516/locations/us-central1/tuningJobs/1206289746024726528
Job state: 2


#### Testing our fine-tuned `gemini-2.5-flash-lite` model

In [6]:
from google import genai
from google.genai import types

Option 1: Using the google-genai SDK (Direct & Recommended)

In [9]:
import vertexai
from vertexai.tuning import sft
from google import genai
from google.genai import types

# ----------------------------------------------------
# 1. Retrieve the fine-tuned model name / endpoint
# ----------------------------------------------------
PROJECT_ID = "starlit-primacy-480312-u9"
REGION = "us-central1"
JOB_ID = "1206289746024726528"

vertexai.init(project=PROJECT_ID, location=REGION)

# Retrieve the job directly by its full resource name
tuning_job = sft.SupervisedTuningJob(
    f"projects/{PROJECT_ID}/locations/{REGION}/tuningJobs/{JOB_ID}"
)

# Get the tuned model endpoint
fine_tuned_model_name = tuning_job.tuned_model_endpoint_name
print(f"Fine-tuned model endpoint: {fine_tuned_model_name}")

# ----------------------------------------------------
# 2. Initialize GenAI Client
# ----------------------------------------------------
client = genai.Client(
    enterprise=True,
    project=PROJECT_ID,
    location=REGION
)

# ----------------------------------------------------
# 3. Inference & Evaluation Functions
# ----------------------------------------------------
def test_messages_for(item):
    return f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"

def gemini_flash_lite_fine_tuned(item):
    response = client.models.generate_content(
        model=fine_tuned_model_name,
        contents=test_messages_for(item),
        config=types.GenerateContentConfig(
            max_output_tokens=7,
            temperature=0.0
        )
    )
    return response.text.strip()

# ----------------------------------------------------
# 4. Test & Evaluate
# ----------------------------------------------------
print(f"Actual price: ${test[0].price:.2f}")
print(f"Predicted: {gemini_flash_lite_fine_tuned(test[0])}")

evaluate(gemini_flash_lite_fine_tuned, test)

2026-08-22 17:18:05,674 INFO google_genai.models AFC is enabled with max remote calls: 10.


Fine-tuned model endpoint: projects/966750849516/locations/us-central1/endpoints/7031073944174067712
Actual price: $144.96


2026-08-22 17:18:11,876 INFO httpx HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/966750849516/locations/us-central1/endpoints/7031073944174067712:generateContent "HTTP/1.1 200 OK"
2026-08-22 17:18:11,922 INFO google_genai.models AFC is enabled with max remote calls: 10.
2026-08-22 17:18:11,926 INFO google_genai.models AFC is enabled with max remote calls: 10.
2026-08-22 17:18:11,946 INFO google_genai.models AFC is enabled with max remote calls: 10.
2026-08-22 17:18:11,950 INFO google_genai.models AFC is enabled with max remote calls: 10.
2026-08-22 17:18:11,955 INFO google_genai.models AFC is enabled with max remote calls: 10.


Predicted: $250


  0%|          | 0/200 [00:00<?, ?it/s]

...

$17 $19 $11 $381 

2026-08-22 17:18:50,578 INFO httpx HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/966750849516/locations/us-central1/endpoints/7031073944174067712:generateContent "HTTP/1.1 200 OK"
2026-08-22 17:18:50,643 INFO httpx HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/966750849516/locations/us-central1/endpoints/7031073944174067712:generateContent "HTTP/1.1 200 OK"


$30 $14 

2026-08-22 17:18:50,987 INFO httpx HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/966750849516/locations/us-central1/endpoints/7031073944174067712:generateContent "HTTP/1.1 200 OK"


$5 

Option 2: Using litellm

In [ ]:
def gemini_flash_lite_fine_tuned_litellm(item):
    prompt = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    
    response = completion(
        model=f"vertex_ai/{fine_tuned_model_name}",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=7,
        temperature=0.0
    )
    return response.choices[0].message.content.strip()

# Test and evaluate
print(f"Actual price: ${test[0].price:.2f}")
print(f"Predicted: {gemini_flash_lite_fine_tuned_litellm(test[0])}")

evaluate(gemini_flash_lite_fine_tuned_litellm, test)